# Converting a Non-Deterministic <span style="font-variant:small-caps;">Fsm</span> into a Deterministic <span style="font-variant:small-caps;">Fsm</span>

In this notebook we show how a non-deterministic <span style="font-variant:small-caps;">Fsm</span>
$$ F = \langle Q, \Sigma, \delta, q_0, A \rangle $$
can be transformed into a deterministic <span style="font-variant:small-caps;">Fsm</span> $\texttt{det}(F)$ such that both <span style="font-variant:small-caps;">Fsm</span>s accept the
same language, that is we have
$$ L(F) = L\bigl(\texttt{det}(F)\bigr). $$
The idea behind this transformation is that the <span style="font-variant:small-caps;">Fsm</span> $\texttt{det}(F)$ has to 
compute the set of all states that the <span style="font-variant:small-caps;">Fsm</span> $F$ could be in. 
Hence the states of the deterministic <span style="font-variant:small-caps;">Fsm</span> $\texttt{det}(F)$ are 
**sets** of states of the non-deterministic <span style="font-variant:small-caps;">Fsm</span> $F$.  A set of these states contains all those states that the non-deterministic <span style="font-variant:small-caps;">Fsm</span> 
$F$ could have reached.  Furthermore, a set $M$ of states of the <span style="font-variant:small-caps;">Fsm</span> $F$ is an accepting state of the <span style="font-variant:small-caps;">Fsm</span> $\texttt{det}(F)$ if the set $M$ contains an accepting state of the <span style="font-variant:small-caps;">Fsm</span> $F$.

In [ ]:
#load "AutomataTypes.fsx"

In [ ]:
open AutomataTypes

## Declaring the necessary types
`Nfa<'state>` and `Dfa<'state>` are records containing the five components of an automaton. State types are generic and must support comparison, because immutable `Set` and `Map` use ordered keys. An NFA transition maps `(state, Eps)` or `(state, Input c)` to a set of states. A DFA transition maps `(state, c)` to a single state. Subset construction returns `Dfa<Set<'state>>`: an F# set can itself be a state.

In order to present the construction of $\texttt{det}(F)$ we first have to define three auxiliary functions.
We start with the function `bigUnion`.  Given a set `M` that contains sets, the epression `bigUnion(M)`
returns the union of all sets in `M`, i.e. we have
$$ \texttt{bigUnion}(M) = \bigcup M = \bigl\{ x \bigm| \exists A \in M: x \in A \bigr\}. $$
F# sets are immutable.

In [ ]:
let bigUnion sets = Set.unionMany sets

In [ ]:
bigUnion [set [1;2;3]; set [2;3;4]; set [3;4;5]]

The function `epsClosure` takes two arguments:
- `s` is a state, 
- `δ` is the transition function of the <span style="font-variant:small-caps;">Nfa</span> $F$.

The function computes the set of all those states that can be reached from the state
`s` via $\varepsilon$-transitions.
Formally, the set $\texttt{epsClosure}(q)$ is defined inductively:
- $s \in \texttt{epsClosure}(s)$.
- $p \in \texttt{epsClosure}(s) \wedge r \in \delta(p, \varepsilon) \;\rightarrow\; r \in \texttt{epsClosure}(s)$.
 
  If the state $p$ is an element of the $\varepsilon$-closure of the state $s$ 
  and there is an $\varepsilon$-transition from $p$ to some state $r$, then $r$ 
  is also an element of the $\varepsilon$-transition of $s$.
  
The implementation of `epsClosure` uses a *fixed-point algorithm*.

In [ ]:
let epsClosure state (delta: Map<'state * Symbol, Set<'state>>) : Set<'state> =
    let rec expand reached =
        let newStates =
            reached
            |> Seq.map (fun q -> Map.tryFind (q, Eps) delta |> Option.defaultValue Set.empty)
            |> bigUnion
        if Set.isSubset newStates reached then reached
        else expand (Set.union reached newStates)
    expand (Set.singleton state)

In order to transform a non-deterministic <span style="font-variant:small-caps;">Fsm</span> $F$ into a deterministic 
<span style="font-variant:small-caps;">Fsm</span>
$\texttt{det}(F)$ we have to extend the function $\delta:Q \times \Sigma \rightarrow 2^Q$ into the function
$$\widehat{\delta}: Q \times \Sigma \rightarrow 2^Q. $$
The idea is that given a state $q$ and a character $c$,  the value of $\widehat{\delta}(q,c)$ is the set of all states that the
<span style="font-variant:small-caps;">Fsm</span> $F$ could reach when it reads the character $c$ in state $q$ and then performs an arbitrary number of $\varepsilon$-transitions.  Formally, the definition of $\widehat{\delta}$ is as follows:
$$ \widehat{\delta}(q_1, c) := \bigcup \bigl\{ \texttt{epsClosure}(q_2) \bigm| q_2 \in \delta(q_1, c) \bigr \}. $$
This formula is to be read as follows:
- For every state $q_2 \in Q$ that can be reached from the state $q_1$ by reading the character $c$ we
  compute $\texttt{epsClosure}(q_2)$.
- Then we take the union of all these sets $\texttt{epsClosure}(q_2)$.

The function $\widehat{\delta}$ is implemented as the function `deltaHat`, which takes three arguments:
- `s` is a state,
- `c` is a character,
- `𝛿` is the transition function of a non-deterministic 
  <span style="font-variant:small-caps;">Fsm</span>.

This function computes the set of all those states that can be reached 
from `s` when we first have a transition from state `s` to some state `p` 
on reading the character `c` followed by any number of $\varepsilon$-transitions
starting in `p`.

In [ ]:
let deltaHat state character delta =
    Map.tryFind (state, Input character) delta
    |> Option.defaultValue Set.empty
    |> Seq.map (fun q -> epsClosure q delta)
    |> bigUnion

let capitalDelta states character delta =
    states |> Seq.map (fun q -> deltaHat q character delta) |> bigUnion

The function  $\widehat{\delta}$ maps a state into a set of states.  Since the <span style="font-variant:small-caps;">Fsm</span> $\texttt{det}(F)$ uses sets of states of the <span style="font-variant:small-caps;">Fsm</span> $F$ as its states we need a function that maps sets of states of the <span style="font-variant:small-caps;">Fsm</span> $F$ into sets of states.  Hence we generalize 
the function $\widehat{\delta}$ to the function
$$ \Delta: 2^Q \times \Sigma \rightarrow 2^Q $$
such that for a set $M$ of states and a character $c$ the expression $\Delta(M, c)$
computes the set of all those states that the <span style="font-variant:small-caps;">Fsm</span> $F$ could be in if it is in a state from the set $M$, then
reads the character $c$, and finally makes some $\varepsilon$-transitions.
The formal definition is as follows: 
$$ \Delta(M,c) := \bigcup \bigl\{ \widehat{\delta}(q,c) \bigm| q \in M \bigr\}. $$
This formula is easy to understand:  For every state  $q \in M$ we compute the set of states that the
<span style="font-variant:small-caps;">Fsm</span> could be in after reading the character $c$ and doing some 
$\varepsilon$-transitions.  Then we take the union of these sets.

The second function above implements $\Delta(M,c)$ by taking the union over the states in `M`.

The function `allStates` takes three arguments:
- $Q$ is $\texttt{epsClosure}(q_0)$, where $q_0$ is the start state of the deterministic <span style="font-variant:small-caps;">Fsm</span> $\texttt{det}(F)$,
- $\delta$ is the transition function of the non-deterministic <span style="font-variant:small-caps;">Fsm</span> $F$, and
- $\Sigma$ is the alphabet of the non-deterministic <span style="font-variant:small-caps;">Fsm</span> $F$.

The function `allStates` computes the set of all states of the deterministic <span style="font-variant:small-caps;">Fsm</span> $\texttt{det}(F)$
that can be reached from the start state.

In [ ]:
let allStates start delta alphabet =
    let rec expand reached =
        let newStates =
            seq { for states in reached do
                      for character in alphabet do
                          yield capitalDelta states character delta }
            |> Set.ofSeq
        if Set.isSubset newStates reached then reached
        else expand (Set.union reached newStates)
    expand (Set.singleton start)

Now we are ready to formally define how the deterministic <span style="font-variant:small-caps;">Fsm</span> $\texttt{det}(F)$
is computed from the non-deterministic <span style="font-variant:small-caps;">Fsm</span>
$F = \bigl\langle Q, \Sigma, \delta, q_0, A \bigr\rangle$.
We define: 
$$ \texttt{det}(F) := \bigl\langle \texttt{allStates}(\texttt{epsClosure}(q_0)), \Sigma, \Delta, \texttt{epsClosure}(q_0), \widehat{A} \bigr\rangle $$
where the components of this tuple are given as follows:
- The set of states of $\texttt{det}(F)$ is the set of all states that can be reached from the set $\texttt{epsClosure}(q_0)$.
- The input alphabet $\Sigma$ does not change when going from $F$ to $\texttt{det}(F)$.
  After all, the deterministic <span style="font-variant:small-caps;">Fsm</span> $\texttt{det}(F)$ has to recognize the same language as the non-deterministic
  <span style="font-variant:small-caps;">Fsm</span> $F$.
- The function $\Delta$, that has been defined previously, specified how the set of states change when a
  character is read.
- The start state $\texttt{epsClosure}(q_0)$ of the non-deterministic <span style="font-variant:small-caps;">Fsm</span> $\texttt{det}(F)$ is the set of all states
  that can be reached from the start state $q_0$ of the non-deterministic <span style="font-variant:small-caps;">Fsm</span> $F$
  via $\varepsilon$-transitions.
- The set of accepting states $\widehat{A}$ is the set of those subsets of $Q$ that contain an accepting
  state of the <span style="font-variant:small-caps;">Fsm</span> $F$:
  $$\widehat{A} := \bigl\{ M \in 2^Q \mid M \cap A \not= \{\} \bigl\}. $$

In [ ]:
let nfa2dfa (nfa: Nfa<'state>) : Dfa<Set<'state>> =
    let start = epsClosure nfa.Start nfa.Transitions
    let states = allStates start nfa.Transitions nfa.Alphabet
    let transitions =
        seq { for source in states do
                  for character in nfa.Alphabet do
                      yield (source, character), capitalDelta source character nfa.Transitions }
        |> Map.ofSeq
    let accepting = states |> Set.filter (fun states -> not (Set.isEmpty (Set.intersect states nfa.Accepting)))
    { States = states; Alphabet = nfa.Alphabet; Transitions = transitions
      Start = start; Accepting = accepting }

The empty subset is a rejecting sink whenever reached. Its transitions are included, so the result is complete over the alphabet. Epsilon cycles terminate because each iteration only adds states to a finite set.

For executable examples we also define language membership, directly from the extended transition relation.

In [ ]:
let acceptsNfa (nfa: Nfa<'state>) (word: string) : bool =
    let start = epsClosure nfa.Start nfa.Transitions
    let finish = word |> Seq.fold (fun states c -> capitalDelta states c nfa.Transitions) start
    not (Set.isEmpty (Set.intersect finish nfa.Accepting))

let acceptsDfa (dfa: Dfa<'state>) (word: string) : bool =
    // A missing transition rejects the word, so partial DFAs are supported too.
    let finish =
        word |> Seq.fold (fun state c -> state |> Option.bind (fun q -> Map.tryFind (q, c) dfa.Transitions)) (Some dfa.Start)
    finish |> Option.exists (fun q -> Set.contains q dfa.Accepting)